In [3]:
from elasticsearch import Elasticsearch

es = Elasticsearch('http://localhost:9200')
print(es.info())

{'name': '473fbd3889bc', 'cluster_name': 'docker-cluster', 'cluster_uuid': '7KOoc0V7Sh2FPelmVBu6Sw', 'version': {'number': '8.13.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '09df99393193b2c53d92899662a8b8b3c55b45cd', 'build_date': '2024-03-22T03:35:46.757803203Z', 'build_snapshot': False, 'lucene_version': '9.10.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [5]:
es.get(index="my-index",id="11")["_source"]

{'title': 'Healthy Living Guide',
 'author': 'Dr. Ramesh Kumar',
 'genre': 'health',
 'year': 2016,
 'rating': 4.0,
 'description': 'A complete guide to a healthy lifestyle.',
 'tags': ['health', 'fitness', 'wellness'],
 'published_at': '2016-01-01'}

In [33]:
result = es.search(
    index='my-index',
    query = {
        "multi_match": {
            "query": " Dr. Ramesh Kumar",
            'fields': ["author", "genre", "description"]
        }
    },
    source=["title"]
)
print(result['hits']['total']['value'])
print([(result['_source'], result['_score']) for result in result['hits']['hits']])

1
[({'title': 'Healthy Living Guide'}, 6.1395006)]


In [41]:
result = es.search(
    index='my-index',
    query={
        "range": {
            "year": {
                "gte": 2014,
                "lt": 2020
            }
        },
        "range":{
            "rating":{
                "gte":5
            }
        }
    }
)
print(result['hits']['total']['value'])
print([(result['_source'], result['_score']) for result in result['hits']['hits']])

3
[({'title': 'my-first-book', 'author': 'venky', 'genre': 'comic', 'year': 2002, 'rating': 6.9, 'description': 'feel good book with self development', 'tags': ['book', 'venky', 'comic', 'self-help'], 'published_at': '2002-05-20'}, 1.0), ({'title': 'my-first-book', 'author': 'venky', 'genre': 'comic', 'rating': 6.9, 'description': 'feel good book with self development', 'tags': ['book', 'venky', 'comic', 'self-help'], 'published_at': '2002-05-20'}, 1.0), ({'title': 'my-first-book', 'author': 'venky', 'rating': 6.9, 'genre': 'comic', 'description': 'feel good book with self development', 'tags': ['book', 'venky', 'comic', 'self-help'], 'year': 2002, 'published_at': '2002-05-20'}, 1.0)]


In [42]:
result = es.search(
    index="my-index",
    query={"match_all":{}}
)

In [43]:
result['hits']['total']

{'value': 17, 'relation': 'eq'}

In [10]:
result = es.search(
    index='my-index',
    aggs={
        'avg_rating': {'avg': {'field': 'rating'}},
        'max_rating': {'max': {'field': 'rating'}},
        'min_rating': {'min' : {'field': 'rating'}},
        'total_books': {'value_count': {'field': 'rating'}},
        'rating_stats': {'stats': {'field': 'rating'}}
    },
    size=0
)

In [11]:
result.keys()

dict_keys(['took', 'timed_out', '_shards', 'hits', 'aggregations'])

In [12]:
result['aggregations']

{'avg_rating': {'value': 4.758823549046236},
 'max_rating': {'value': 6.900000095367432},
 'min_rating': {'value': 3.700000047683716},
 'total_books': {'value': 17},
 'rating_stats': {'count': 17,
  'min': 3.700000047683716,
  'max': 6.900000095367432,
  'avg': 4.758823549046236,
  'sum': 80.90000033378601}}